# CCA: from code observations to a compiler change

CCA is Merlin's **Common Compute Abstraction**. We will decode a short instruction
listing, compare its properties with another listing, and inspect the compiler
changes suggested by the real action catalog.

The listings are authored teaching inputs, with illustrative addresses and no
encoding bytes. We will inspect code structure; we will not compile or run a kernel.

Use the [tutorial launcher](README.md) for setup. The [CCA guide](../../docs/guides/11-optimization.md)
connects this example to the actual agent loops.

**Step 5 of 5: CCA** | [All tutorials](README.md) | [Previous: Transformations](04_transformations.ipynb)


In [ ]:
from collections import Counter
from dataclasses import asdict, replace
from pprint import pprint

from merlin.kernels.cca import lift_asm
from merlin.kernels.cca_compare import compare, compare_evidence, uncomparable_axes
from merlin.kernels.decode.rvv import decode_text
from merlin.kernels.action_catalog import (
    achieved_residual, build_catalog, seam_location,
)

## 1. Decode a short listing

FMA means fused multiply-add. LMUL is RVV's vector register-group multiplier.
The decoder reads the explicit `e32, m1/m2/...` configuration; the lifter summarizes
arithmetic form, register grouping and accumulator use.

`op="matmul"` is a label we supply for this demonstration. The little loop below
is not a complete matrix multiplication implementation.

In [ ]:
def listing(*, fused=True, lmul="m2", include_vtype=True):
    instructions = [f"vsetivli zero, 4, e32, {lmul}, ta, ma"] if include_vtype else []
    instructions += ["vle32.v v8, (a1)", "vmv.v.i v16, 0"]
    loop_pc = 4 * len(instructions)
    instructions += (
        ["vfmacc.vf v16, fa0, v8"] if fused else
        ["vfmul.vf v12, v8, fa0", "vfadd.vv v16, v16, v12"]
    )
    instructions += [
        "addi t0, t0, -1",
        f"bnez t0, {loop_pc:x} <tutorial_kernel+0x{loop_pc:x}>",
        "vse32.v v16, (a2)",
        "ret",
    ]
    return "0000000000000000 <tutorial_kernel>:\n" + "\n".join(
        f"{4 * index:x}: {instruction}" for index, instruction in enumerate(instructions)
    )


def observe(text, label):
    return lift_asm(decode_text(text), op="matmul", source=f"authored tutorial: {label}")


reference_text = listing(fused=True, lmul="m2")
baseline_text = listing(fused=False, lmul="m1")
print(reference_text)

In [ ]:
reference = observe(reference_text, "reference pattern; not qualified expert code")
baseline = observe(baseline_text, "candidate baseline pattern")


def summary(record):
    return {
        "scope": record.scope,
        "contraction": record.compute.contraction_form,
        "accumulator": record.compute.accumulator_dtype,
        "accumulator_resident": record.compute.accumulator_resident,
        "LMUL": record.vector.lmul if record.vector else None,
    }


pprint({"reference": summary(reference), "candidate": summary(baseline)})

## 2. Compare and find an edit

A divergence names a property, such as `vector.lmul`, and its two values.
`uncomparable_axes` tells us where one listing leaves a value unknown.
The action catalog maps a divergence to an edit mechanism and source file.

Changing separate floating multiply/add to FMA can change rounding. Its proposed
edit still needs to satisfy the program's numerical contract.

In [ ]:
gaps = compare(reference, baseline)
pprint([asdict(gap) for gap in gaps])
print("One-sided missing values:", uncomparable_axes(reference, baseline))

actions, unrouted = build_catalog(gaps)
pprint([
    {
        "axis": action.divergence_axis,
        "change_class": action.action_class,
        "seam": action.target_seam,
        "source": seam_location(action.target_seam, backend=action.backend)["seam_file"],
        "intended": action.intended_facet,
        "forkable_now": action.forkable_now,
    }
    for action in actions
])
print("Unrouted axes:", [gap.axis for gap in unrouted])

## 3. Ask what evidence is missing

The vector lifter fills structural values, but it does not attach the per-axis
witness records required by `compare_evidence`. Try the stricter comparison below.
Its denominator counts the axes it could actually compare.

In [ ]:
supported = compare_evidence(reference, baseline)
print("Status:", supported.status)
print("Compared-axis denominator:", supported.denominator)
print("Unknown reasons:", dict(Counter(row["reason"] for row in supported.unknowns)))
pprint([row for row in supported.unknowns if row["axis"] == "vector.lmul"])

A kernel and a whole program answer different questions. The comparator refuses
to compare their scopes, even when they use the same field names.

In [ ]:
try:
    compare(reference, replace(baseline, scope="program"))
except ValueError as error:
    print(error)

## 4. Check the proposed edit's promise

A real search rebuilds the compiler and inspects its emitted code. Here we author
a matching listing to see how the promise check works.

In [ ]:
candidate = observe(listing(fused=True, lmul="m2"), "matching candidate pattern")
pprint([
    {
        "axis": action.divergence_axis,
        "has_checkable_promise": bool(action.intended_facet),
        "remaining_promised_axes": achieved_residual(action, candidate),
    }
    for action in actions
])
print("Structural gaps after the authored change:", [gap.axis for gap in compare(reference, candidate)])
print("Numerical correctness: not executed. Performance: not measured.")

An empty residual can mean either that the promise was met or that the action had
no checkable promise. The real beam distinguishes those cases in `make_step`.
Output correctness and speed still come from execution, not from this comparison.

## 5. Try a different candidate

Change the arithmetic or vector group. Then remove the vector configuration and
watch which values become unknown. The table uses Merlin's comparator and action
catalog; it adds no score or optimization policy.

In [ ]:
def explore(fused=False, lmul="m1", include_vtype=True):
    from dataclasses import is_dataclass
    from html import escape
    from IPython.display import HTML, display

    observed = observe(listing(fused=fused, lmul=lmul, include_vtype=include_vtype), "interactive candidate")
    differences = compare(reference, observed)
    proposals, missing_routes = build_catalog(differences)
    missing = uncomparable_axes(reference, observed)
    strict = compare_evidence(reference, observed)
    by_axis = {action.divergence_axis: action for action in proposals}
    gap_axes = {gap.axis for gap in differences}
    missing_sides = dict(missing)
    axes = list(dict.fromkeys(["compute.contraction_form", "vector.lmul", *sorted(gap_axes), *missing_sides]))

    def value(record, axis):
        result = record
        for part in axis.split("."):
            result = getattr(result, part, None)
        return "unknown" if result is None else "present" if is_dataclass(result) else str(result)

    rows = []
    for axis in axes:
        action = by_axis.get(axis)
        result = "divergent" if axis in gap_axes else (
            "missing in " + missing_sides[axis] if axis in missing_sides else "no reported divergence"
        )
        edit = action.action_class + ": " + action.target_seam if action else (
            "no route" if axis in gap_axes else ""
        )
        cells = [axis, value(reference, axis), value(observed, axis), result, edit]
        rows.append("<tr>" + "".join("<td>" + escape(item) + "</td>" for item in cells) + "</tr>")
    headers = "<tr>" + "".join("<th>" + title + "</th>" for title in (
        "Property", "Reference", "Candidate", "Comparison", "Proposed edit",
    )) + "</tr>"
    unknown_text = "; ".join(axis + " (" + side + " missing)" for axis, side in missing) or "none"
    display(HTML(
        "<style>.cca-table td,.cca-table th{padding:7px;text-align:left;vertical-align:top;border-bottom:1px solid #aaa}"
        ".cca-table{border-collapse:collapse;width:100%;font-size:14px}</style>"
        "<table class='cca-table'>" + headers + "".join(rows) + "</table>"
        "<p><b>One-sided unknowns:</b> " + escape(unknown_text) + "</p>"
        "<p><b>Evidence-aware comparison:</b> " + escape(strict.status)
        + "; compared axes: " + str(strict.denominator) + ".</p>"
    ))

explore()

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional controls are unavailable; call explore(fused=True, lmul='m2') above.")
else:
    from IPython.display import display

    controls = widgets.interactive(
        explore,
        fused=widgets.Checkbox(value=False, description="Use FMA"),
        lmul=widgets.Dropdown(options=["m1", "m2", "m4"], value="m1", description="LMUL"),
        include_vtype=widgets.Checkbox(value=True, description="Include vtype"),
    )
    display(controls)

## Where this fits

The mining beam compares actual emitted code, proposes changes, certifies builds
and checks whether the intended properties appeared. Phase 1 also has an oracle-free
CCA broker for schema/action queries. Explicit Phase 2 component feedback binds
its observations to selected artifacts and runtime providers.

The [CCA guide](../../docs/guides/11-optimization.md) points to those owners. Return to the
[tutorial catalog](README.md) for the rest of the compilation flow.

**Step 5 of 5: CCA** | [All tutorials](README.md) | [Previous: Transformations](04_transformations.ipynb)
